In [ ]:
import numpy as np
import pandas as pd
import warnings
import gc

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")


(297236, 18) (99639, 17)
Target mean: 0.0290812687561399
(297236, 49) (99639, 49)
After TE: (297236, 59)

FOLD 1
LGB AUC: 0.970827191521645
CAT AUC: 0.9706697505798596

FOLD 2
LGB AUC: 0.9717518017528087
CAT AUC: 0.9719182671730793

FOLD 3
LGB AUC: 0.9712128198908753
CAT AUC: 0.971297463900337

FOLD 4
LGB AUC: 0.9719858390481985
CAT AUC: 0.971510451756512

FOLD 5
LGB AUC: 0.9726525296173367
CAT AUC: 0.973227441509707

OOF LGB: 0.9716685117372879
OOF CAT: 0.9717034088651697

BEST BLEND
Best OOF AUC: 0.9724936936792249
Best LGB weight: 0.51
Best CAT weight: 0.49
Saved submission_best_auc_blend.csv
Saved extra files:
submission_blend_040_060.csv
submission_blend_035_065.csv
submission_cat_avg.csv
submission_lgb_avg.csv

F1 ANALYSIS ONLY
Best F1: 0.5132889508179801
Best threshold: 0.25

SUBMIT FIRST:
submission_best_auc_blend.csv


In [ ]:

# LOAD DATA

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

y = train["order_placed"].astype(int)
test_ids = test["id"]

print(train.shape, test.shape)
print("Target mean:", y.mean())



In [ ]:

# FEATURE ENGINEERING

def make_features(df):
    df = df.copy()

    for c in ["f3", "f4", "f5"]:
        df[c] = pd.to_datetime(df[c], errors="coerce")

    df["session_duration"] = (df["f4"] - df["f3"]).dt.total_seconds()
    df["active_time"] = (df["f5"] - df["f3"]).dt.total_seconds()
    df["idle_time"] = (df["f4"] - df["f5"]).dt.total_seconds()

    df["date"] = df["f3"].dt.strftime("%Y-%m-%d")
    df["hour"] = df["f3"].dt.hour
    df["dow"] = df["f3"].dt.dayofweek
    df["day"] = df["f3"].dt.day

    df["hour_sin"] = np.sin(2*np.pi*df["hour"].fillna(0)/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"].fillna(0)/24)

    df["id_mod_2"] = df["id"] % 2
    df["id_mod_5"] = df["id"] % 5
    df["id_mod_10"] = df["id"] % 10
    df["id_mod_100"] = df["id"] % 100

    df["has_cart"] = (df["f10"] > 0).astype(int)
    df["has_value"] = (df["f11"] > 0).astype(int)
    df["accepted"] = (df["f17"].astype(str) == "ACCEPTED").astype(int)
    df["ignored"] = (df["f17"].astype(str) == "IGNORED").astype(int)
    df["declined"] = (df["f17"].astype(str) == "DECLINED").astype(int)

    df["conversion_signal"] = df["has_cart"] + df["has_value"] + df["accepted"]

    df["meets_min"] = (df["f11"] >= df["f14"]).astype(int)
    df["final_intent"] = df["meets_min"] + df["accepted"] + (df["f10"] > 2).astype(int) + (df["f13"] > 0).astype(int)

    df["cart_quality"] = df["f11"] / (df["f10"] + 1)
    df["cart_to_min"] = df["f11"] / (df["f14"] + 1)
    df["discount_to_cart"] = df["f13"] / (df["f11"] + 1)
    df["discount_to_min"] = df["f13"] / (df["f14"] + 1)
    df["offers_per_decline"] = df["f15"] / (df["f8"] + 1)
    df["items_per_offer"] = df["f10"] / (df["f15"] + 1)
    df["value_per_offer"] = df["f11"] / (df["f15"] + 1)
    df["urgency"] = df["f10"] / (df["session_duration"] + 1)
    df["value_speed"] = df["f11"] / (df["session_duration"] + 1)

    df["promo_resp"] = df["f12"].astype(str) + "_" + df["f17"].astype(str)
    df["cust_promo"] = df["f9"].astype(str) + "_" + df["f12"].astype(str)
    df["cust_resp"] = df["f9"].astype(str) + "_" + df["f17"].astype(str)
    df["action_resp"] = df["f7"].astype(str) + "_" + df["f17"].astype(str)
    df["date_promo"] = df["date"].astype(str) + "_" + df["f12"].astype(str)
    df["date_resp"] = df["date"].astype(str) + "_" + df["f17"].astype(str)

    cat_cols = [
        "f6","f7","f9","f12","f17","date",
        "promo_resp","cust_promo","cust_resp",
        "action_resp","date_promo","date_resp"
    ]

    for c in cat_cols:
        df[c] = df[c].astype(str).fillna("missing")

    df = df.drop(columns=["f2", "f3", "f4", "f5"], errors="ignore")
    return df

X = make_features(train.drop(columns=["order_placed"]))
X_test = make_features(test)

cat_cols = [
    "f6","f7","f9","f12","f17","date",
    "promo_resp","cust_promo","cust_resp",
    "action_resp","date_promo","date_resp"
]

cat_cols = [c for c in cat_cols if c in X.columns]

print(X.shape, X_test.shape)


In [ ]:

# TARGET ENCODING

def add_cv_target_encoding(X, X_test, y, cols, n_splits=5):
    X = X.copy()
    X_test = X_test.copy()

    global_mean = y.mean()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for col in cols:
        te_name = col + "_te"
        X[te_name] = global_mean
        X_test[te_name] = 0

        for tr_idx, val_idx in skf.split(X, y):
            tmp = pd.DataFrame({
                col: X.iloc[tr_idx][col],
                "target": y.iloc[tr_idx]
            })

            means = tmp.groupby(col)["target"].mean()
            X.loc[X.index[val_idx], te_name] = X.iloc[val_idx][col].map(means).fillna(global_mean)

        full_means = pd.DataFrame({
            col: X[col],
            "target": y
        }).groupby(col)["target"].mean()

        X_test[te_name] = X_test[col].map(full_means).fillna(global_mean)

    return X, X_test

te_cols = [
    "date",
    "f7",
    "f9",
    "f12",
    "f17",
    "promo_resp",
    "cust_promo",
    "cust_resp",
    "date_promo",
    "date_resp"
]

te_cols = [c for c in te_cols if c in X.columns]

X, X_test = add_cv_target_encoding(X, X_test, y, te_cols)

print("After TE:", X.shape)

# =========================
# MODEL PARAMETERS
# =========================
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "n_estimators": 2500,
    "learning_rate": 0.018,
    "num_leaves": 96,
    "min_child_samples": 45,
    "subsample": 0.88,
    "colsample_bytree": 0.88,
    "reg_alpha": 0.3,
    "reg_lambda": 1.5,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1
}

cat_params = {
    "iterations": 2500,
    "learning_rate": 0.018,
    "depth": 7,
    "l2_leaf_reg": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": False
}


In [ ]:
# TRAINING
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

pred_lgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\nFOLD {fold}")

    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]

    X_tr_lgb = X_tr.copy()
    X_val_lgb = X_val.copy()
    X_test_lgb = X_test.copy()

    for c in cat_cols:
        X_tr_lgb[c] = X_tr_lgb[c].astype("category")
        X_val_lgb[c] = X_val_lgb[c].astype("category")
        X_test_lgb[c] = X_test_lgb[c].astype("category")

    lgb = LGBMClassifier(**lgb_params)
    lgb.fit(X_tr_lgb, y_tr)

    oof_lgb[val_idx] = lgb.predict_proba(X_val_lgb)[:, 1]
    pred_lgb += lgb.predict_proba(X_test_lgb)[:, 1] / 5

    print("LGB AUC:", roc_auc_score(y_val, oof_lgb[val_idx]))

    cat = CatBoostClassifier(**cat_params)
    cat.fit(
        X_tr, y_tr,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        use_best_model=True,
        verbose=False
    )

    oof_cat[val_idx] = cat.predict_proba(X_val)[:, 1]
    pred_cat += cat.predict_proba(X_test)[:, 1] / 5

    print("CAT AUC:", roc_auc_score(y_val, oof_cat[val_idx]))

    gc.collect()

print("\nOOF LGB:", roc_auc_score(y, oof_lgb))
print("OOF CAT:", roc_auc_score(y, oof_cat))


In [ ]:


# BEST AUC BLEND SEARCH

best_auc = -1
best_w = None

for w in np.arange(0.00, 1.01, 0.01):
    oof_blend = w * oof_lgb + (1 - w) * oof_cat
    auc = roc_auc_score(y, oof_blend)

    if auc > best_auc:
        best_auc = auc
        best_w = w

print("\nBEST BLEND")
print("Best OOF AUC:", best_auc)
print("Best LGB weight:", best_w)
print("Best CAT weight:", 1 - best_w)

final_pred = best_w * pred_lgb + (1 - best_w) * pred_cat

sub_best_auc = pd.DataFrame({
    "id": test_ids,
    "order_placed": final_pred
})

sub_best_auc.to_csv("submission_best_auc_blend.csv", index=False)
print("Saved submission_best_auc_blend.csv")



In [ ]:

# EXTRA SAFE FILES

sub_040_060 = pd.DataFrame({
    "id": test_ids,
    "order_placed": 0.40 * pred_lgb + 0.60 * pred_cat
})

sub_035_065 = pd.DataFrame({
    "id": test_ids,
    "order_placed": 0.35 * pred_lgb + 0.65 * pred_cat
})

sub_cat = pd.DataFrame({
    "id": test_ids,
    "order_placed": pred_cat
})

sub_lgb = pd.DataFrame({
    "id": test_ids,
    "order_placed": pred_lgb
})

sub_040_060.to_csv("submission_blend_040_060.csv", index=False)
sub_035_065.to_csv("submission_blend_035_065.csv", index=False)
sub_cat.to_csv("submission_cat_avg.csv", index=False)
sub_lgb.to_csv("submission_lgb_avg.csv", index=False)

print("Saved extra files:")
print("submission_blend_040_060.csv")
print("submission_blend_035_065.csv")
print("submission_cat_avg.csv")
print("submission_lgb_avg.csv")



In [ ]:

# F1 THRESHOLD CALCULATION ONLY
# DO NOT USE FOR AUC SUBMISSION

best_f1 = 0
best_thr = 0.5

best_oof = best_w * oof_lgb + (1 - best_w) * oof_cat

for t in np.arange(0.01, 1.00, 0.01):
    f1 = f1_score(y, (best_oof >= t).astype(int))

    if f1 > best_f1:
        best_f1 = f1
        best_thr = t

print("\nF1 ANALYSIS ONLY")
print("Best F1:", best_f1)
print("Best threshold:", best_thr)

print("\nSUBMIT FIRST:")
print("submission_best_auc_blend.csv")

In [ ]:

# SAFE BEST (your current best)

sub_blend_045 = pd.DataFrame({
    "id": test_ids,
    "order_placed": 0.45 * pred_lgb + 0.55 * pred_cat
})


# NEW TRY (slight improvement attempt)

sub_blend_043 = pd.DataFrame({
    "id": test_ids,
    "order_placed": 0.43 * pred_lgb + 0.57 * pred_cat
})


# INDIVIDUAL MODELS (optional)

sub_cat = pd.DataFrame({
    "id": test_ids,
    "order_placed": pred_cat
})

sub_lgb = pd.DataFrame({
    "id": test_ids,
    "order_placed": pred_lgb
})


# SAVE ALL FILES

sub_blend_045.to_csv("submission_blend_045_055.csv", index=False)
sub_blend_043.to_csv("submission_blend_043_057.csv", index=False)
sub_cat.to_csv("submission_cat_avg.csv", index=False)
sub_lgb.to_csv("submission_lgb_avg.csv", index=False)

print("Saved files:")
print("submission_blend_045_055.csv  ← SAFE")
print("submission_blend_043_057.csv  ← TRY THIS")
print("submission_cat_avg.csv")
print("submission_lgb_avg.csv")

Saved files:
submission_blend_045_055.csv  ← SAFE
submission_blend_043_057.csv  ← TRY THIS
submission_cat_avg.csv
submission_lgb_avg.csv


In [4]:
sub_final = pd.DataFrame({
    "id": test_ids,
    "order_placed": 0.41 * pred_lgb + 0.59 * pred_cat
})

sub_final.to_csv("submission_final_push.csv", index=False)

print("Saved submission_final_push.csv")

Saved submission_final_push.csv


In [5]:
sub_038 = pd.DataFrame({
    "id": test_ids,
    "order_placed": 0.38 * pred_lgb + 0.62 * pred_cat
})

sub_038.to_csv("submission_038_062.csv", index=False)
print("Saved submission_038_062.csv")
sub_037 = pd.DataFrame({
    "id": test_ids,
    "order_placed": 0.37 * pred_lgb + 0.63 * pred_cat
})

sub_037.to_csv("submission_037_063.csv", index=False)
print("Saved submission_037_063.csv")

Saved submission_038_062.csv
Saved submission_037_063.csv
